In [1]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.sql.types import ArrayType, DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")

import pessoal
from pessoal import *

spark.sparkContext.setLogLevel("ERROR")

Tempo inicial da execucao: 2025-11-24 10:36:09.382059
User: tatiane
Node: tatiane-Inspiron-3583


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/24 10:36:13 WARN Utils: Your hostname, tatiane-Inspiron-3583, resolves to a loopback address: 127.0.1.1; using 192.168.0.14 instead (on interface wlo1)
25/11/24 10:36:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/24 10:36:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/24 10:36:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/24 10:36:17 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 45744)
Traceb

## Leitura da base

In [2]:
pnadc_2023_2025_3 = spark.read.parquet("/home/tatiane/Downloads/NINSOC/base_final/pnadc_2023_2025_3/")
pessoal.completudeSchema(pnadc_2023_2025_3)

[Stage 1:===================================================>       (7 + 1) / 8]

Qtd. registros: 5249881 | Quantidade de colunas:  16
root
 |-- ano: integer (nullable = true)
 |-- sexo: string (nullable = true)
 |-- raca_cor: string (nullable = true)
 |-- idade_dt_referencia: integer (nullable = true)
 |-- qtd_pessoa_domicilio: integer (nullable = true)
 |-- condicao_domicilio: string (nullable = true)
 |-- situacao_domicilio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- rmh_todos_trabalhos: double (nullable = true)
 |-- peso_domicilio_pessoa: float (nullable = true)
 |-- tp_remuneracao_habitual: string (nullable = true)
 |-- id_unico: long (nullable = true)
 |-- rendimento_habitual: string (nullable = true)
 |-- recebe_remuneracao: string (nullable = true)
 |-- renda_pc: double (nullable = true)
 |-- ind_pobreza: integer (nullable = true)



In [3]:
# Mesclando raca_cor para que tanto a cor preta quanto a parda sejam identificadas como negra.
pnadc_2023_2025_3 = pnadc_2023_2025_3.withColumn("raca_cor",
    F.when((F.col("raca_cor") == "Preta") | (F.col("raca_cor") == "Parda"), "Negra").otherwise(F.col("raca_cor")))

### Análise descritiva

In [4]:
# Distribuição demográfica básica
var_list = ["sexo", "raca_cor", "uf"]
for col in var_list:
    pnadc_2023_2025_3.groupBy(col).count().orderBy(col).show()

+------+-------+
|  sexo|  count|
+------+-------+
| Homem|2534967|
|Mulher|2714914|
+------+-------+



+--------+-------+
|raca_cor|  count|
+--------+-------+
| Amarela|  26138|
|  Branca|2058331|
|Ignorado|    502|
|Indigena|  30701|
|   Negra|3134209|
+--------+-------+



[Stage 10:==================================================>       (7 + 1) / 8]

+---+------+
| uf| count|
+---+------+
| 11| 85675|
| 12|104344|
| 13|153550|
| 14| 63959|
| 15|191104|
| 16| 47947|
| 17| 80715|
| 21|340202|
| 22|123850|
| 23|248510|
| 24|105996|
| 25|141044|
| 26|213659|
| 27|217456|
| 28| 98411|
| 29|250913|
| 31|389681|
| 32|188204|
| 33|375648|
| 35|416538|
+---+------+
only showing top 20 rows


In [6]:
# Características domiciliares
var_list = ["situacao_domicilio", "qtd_pessoa_domicilio"]
for col in var_list:
    pnadc_2023_2025_3.groupBy(col).count().orderBy(col).show()

+------------------+-------+
|situacao_domicilio|  count|
+------------------+-------+
|             Rural|1404195|
|            Urbana|3845686|
+------------------+-------+



[Stage 22:==================================================>       (7 + 1) / 8]

+--------------------+-------+
|qtd_pessoa_domicilio|  count|
+--------------------+-------+
|                   1| 383457|
|                   2|1178134|
|                   3|1418613|
|                   4|1243140|
|                   5| 595765|
|                   6| 240222|
|                   7| 102382|
|                   8|  46296|
|                   9|  21213|
|                  10|   9990|
|                  11|   4939|
|                  12|   2592|
|                  13|    988|
|                  14|   1050|
|                  15|    345|
|                  16|    224|
|                  17|    119|
|                  18|    252|
|                  19|     95|
|                  21|     21|
+--------------------+-------+
only showing top 20 rows


- Renda habitual média e mediana por sexo, raça e UF

In [7]:
# geral
pnadc_2023_2025_3.select(F.mean("renda_pc"), F.expr("percentile(renda_pc, 0.5)")).show()

[Stage 25:==================================================>       (7 + 1) / 8]

+------------------+----------------------------+
|     avg(renda_pc)|percentile(renda_pc, 0.5, 1)|
+------------------+----------------------------+
|502.57967944794296|                         0.0|
+------------------+----------------------------+



In [8]:
# por sexo
pnadc_2023_2025_3.groupBy("sexo").agg(F.mean("renda_pc"), F.expr("percentile(renda_pc, 0.5)")).show()

[Stage 28:==================================================>       (7 + 1) / 8]

+------+-----------------+----------------------------+
|  sexo|    avg(renda_pc)|percentile(renda_pc, 0.5, 1)|
+------+-----------------+----------------------------+
| Homem|643.0324712913142|                        62.5|
|Mulher|371.4362279858732|                         0.0|
+------+-----------------+----------------------------+



In [4]:
pnadc_2023_2025_3.filter((F.col("sexo") == "Mulher") & (F.col("renda_pc") == 0)).count()

1766430

In [5]:
pnadc_2023_2025_3.filter((F.col("sexo") == "Mulher") & (F.col("renda_pc") > 0)).count()

948484

In [6]:
pnadc_2023_2025_3.filter(F.col("sexo") == "Mulher") \
    .agg(F.mean((F.col("renda_pc") == 0).cast("int")).alias("prop_mulheres_renda_zero")) \
    .show()

[Stage 10:==================================================>       (7 + 1) / 8]

+------------------------+
|prop_mulheres_renda_zero|
+------------------------+
|      0.6506393941023546|
+------------------------+



In [9]:
# por raça/cor
pnadc_2023_2025_3.groupBy("raca_cor").agg(F.mean("renda_pc"),F.expr("percentile(renda_pc, 0.5)")).show()

[Stage 31:==================================================>       (7 + 1) / 8]

+--------+-----------------+----------------------------+
|raca_cor|    avg(renda_pc)|percentile(renda_pc, 0.5, 1)|
+--------+-----------------+----------------------------+
|Ignorado|790.2958546765321|                         0.0|
|   Negra|373.4977262607052|                         0.0|
|Indigena|299.4113199017564|                         0.0|
| Amarela|845.0355588779153|                         0.0|
|  Branca|697.7435019452213|                         0.0|
+--------+-----------------+----------------------------+



In [10]:
# por uf
pnadc_2023_2025_3.groupBy("uf").agg(F.mean("renda_pc")).orderBy("uf").show() 

[Stage 34:==================================================>       (7 + 1) / 8]

+---+------------------+
| uf|     avg(renda_pc)|
+---+------------------+
| 11|482.41299118746423|
| 12| 322.4926836729835|
| 13|291.40623239529253|
| 14| 433.1547284539221|
| 15| 331.8837271516928|
| 16| 421.2793239277267|
| 17| 493.8059323588214|
| 21|204.33234017293452|
| 22|273.20289570106013|
| 23|  261.253483383032|
| 24| 343.0836193946663|
| 25| 317.3255352858682|
| 26| 311.5815179116861|
| 27| 270.8593691610153|
| 28|321.57747274355967|
| 29| 288.1214397316327|
| 31|  549.558530563815|
| 32| 581.9879589405977|
| 33| 666.1074392206218|
| 35| 746.8993407287988|
+---+------------------+
only showing top 20 rows


- Evolução temporal (2019–2025)

In [11]:
# Evolução da renda per capita ao longo do tempo
pnadc_2023_2025_3.groupBy("ano").agg(F.mean("renda_pc")).orderBy("ano").show()

[Stage 37:==================================================>       (7 + 1) / 8]

+----+-----------------+
| ano|    avg(renda_pc)|
+----+-----------------+
|2023|447.4873169042408|
|2024|510.4767356383576|
|2025|564.8991062386722|
+----+-----------------+



In [12]:
# Evolução da pobreza
pnadc_2023_2025_3.groupBy("ano", "ind_pobreza").count().orderBy("ano").show()

[Stage 40:==================================================>       (7 + 1) / 8]

+----+-----------+-------+
| ano|ind_pobreza|  count|
+----+-----------+-------+
|2023|          0| 343495|
|2023|          1|1557494|
|2024|          0| 415663|
|2024|          1|1494784|
|2025|          1|1096185|
|2025|          0| 342260|
+----+-----------+-------+



In [7]:
# Evolução por sexo + raça
pnadc_2023_2025_3.groupBy("ano", "sexo", "raca_cor", "ind_pobreza").count().orderBy("ano").show(200)

[Stage 13:==================================================>       (7 + 1) / 8]

+----+------+--------+-----------+------+
| ano|  sexo|raca_cor|ind_pobreza| count|
+----+------+--------+-----------+------+
|2023|Mulher|   Negra|          1|520563|
|2023|Mulher| Amarela|          0|  1121|
|2023| Homem|   Negra|          1|452368|
|2023|Mulher|Ignorado|          0|    19|
|2023| Homem|  Branca|          0|106535|
|2023| Homem|   Negra|          0|102710|
|2023|Mulher|   Negra|          0| 56886|
|2023|Mulher|  Branca|          1|318213|
|2023| Homem|Ignorado|          1|    62|
|2023| Homem| Amarela|          0|  1351|
|2023|Mulher|  Branca|          0| 73695|
|2023| Homem| Amarela|          1|  3501|
|2023| Homem|Ignorado|          0|    30|
|2023| Homem|  Branca|          1|248125|
|2023|Mulher| Amarela|          1|  4866|
|2023| Homem|Indigena|          1|  4326|
|2023|Mulher|Indigena|          1|  5364|
|2023|Mulher|Ignorado|          1|   106|
|2023| Homem|Indigena|          0|   704|
|2023|Mulher|Indigena|          0|   444|
|2024| Homem| Amarela|          1|

- Cruzadas importantes

In [14]:
#Renda média: mulheres negras vs outros grupos
pnadc_2023_2025_3.groupBy("sexo", "raca_cor").agg(F.mean("renda_pc")).orderBy("sexo", "raca_cor").show()

[Stage 46:==================================================>       (7 + 1) / 8]

+------+--------+------------------+
|  sexo|raca_cor|     avg(renda_pc)|
+------+--------+------------------+
| Homem| Amarela|1079.6157227212284|
| Homem|  Branca|  891.286985836774|
| Homem|Ignorado|1121.1729437229435|
| Homem|Indigena| 387.1356801464648|
| Homem|   Negra| 483.9559313137127|
|Mulher| Amarela| 655.3228486961415|
|Mulher|  Branca| 523.2380347164544|
|Mulher|Ignorado| 482.3256410256411|
|Mulher|Indigena| 222.5130894451921|
|Mulher|   Negra| 267.7763254334305|
+------+--------+------------------+



In [15]:
# Taxa de pobreza: mulheres negras vs outros grupos
pnadc_2023_2025_3.groupBy("ano", "sexo", "raca_cor").agg(F.mean("ind_pobreza")).orderBy("ano", "sexo", "raca_cor").show()

[Stage 49:==================================================>       (7 + 1) / 8]

+----+------+--------+------------------+
| ano|  sexo|raca_cor|  avg(ind_pobreza)|
+----+------+--------+------------------+
|2023| Homem| Amarela| 0.721558120362737|
|2023| Homem|  Branca|0.6996137145435064|
|2023| Homem|Ignorado|0.6739130434782609|
|2023| Homem|Indigena|0.8600397614314115|
|2023| Homem|   Negra|0.8149629421450679|
|2023|Mulher| Amarela|0.8127609821279439|
|2023|Mulher|  Branca|0.8119584188125785|
|2023|Mulher|Ignorado|             0.848|
|2023|Mulher|Indigena|0.9235537190082644|
|2023|Mulher|   Negra|0.9014874040824384|
|2024| Homem| Amarela|0.6481390793339863|
|2024| Homem|  Branca|0.6598532228267318|
|2024| Homem|Ignorado|0.6190476190476191|
|2024| Homem|Indigena|0.8300055980593394|
|2024| Homem|   Negra|0.7723037981037267|
|2024|Mulher| Amarela|0.7583945956685874|
|2024|Mulher|  Branca|0.7784384385911818|
|2024|Mulher|Ignorado|0.7230769230769231|
|2024|Mulher|Indigena|0.9051012145748988|
|2024|Mulher|   Negra|0.8684350522835232|
+----+------+--------+------------

### Análise de probabilidade de pobreza
- Medição de quais grupos têm maior probabilidade de serem pobres, com foco em mulheres negras.

In [16]:
# Comparação simples das taxas de pobreza: Comparar proporções de pessoas pobres entre grupos
prob_pobreza = pnadc_2023_2025_3.groupBy("sexo", "raca_cor").agg(F.avg("ind_pobreza").alias("taxa_pobreza"))
prob_pobreza.show()

[Stage 52:==================================================>       (7 + 1) / 8]

+------+--------+------------------+
|  sexo|raca_cor|      taxa_pobreza|
+------+--------+------------------+
| Homem|Ignorado|0.6487603305785123|
| Homem|  Branca|0.6678255612595165|
| Homem|   Negra|0.7813405429527123|
|Mulher|Indigena|0.9063569682151589|
| Homem| Amarela|0.6692050996834089|
|Mulher| Amarela|0.7718496989827693|
|Mulher|  Branca|0.7853392596643942|
| Homem|Indigena|0.8318806219928875|
|Mulher|   Negra|0.8760529572722584|
|Mulher|Ignorado|0.8038461538461539|
+------+--------+------------------+



- Probabilidade ajustada: regressão logística que estima a probabilidade de pobreza levando em conta o sexo, a raça/cor, a idade, a UF e o tipo de remuneração.

In [9]:
# Transformando variáveis categóricas em vetores (StringIndexer + OneHotEncoder)
cat_cols = ["sexo", "raca_cor", "uf"]
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_vec") for c in cat_cols]

# Criação de vetor de features
features = ["sexo_vec", "raca_cor_vec", "uf_vec", "idade_dt_referencia", "qtd_pessoa_domicilio"]
assembler = VectorAssembler(inputCols=features, outputCol="features")

# Modelo logistico
lr = LogisticRegression(featuresCol="features", labelCol="ind_pobreza")

# Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])
model = pipeline.fit(pnadc_2023_2025_3)

In [27]:
# Extração de probabilidades
pred = model.transform(pnadc_2023_2025_3)
pred.select("sexo", "raca_cor", "probability", "prediction").show(20)

+------+--------+--------------------+----------+
|  sexo|raca_cor|         probability|prediction|
+------+--------+--------------------+----------+
|Mulher|  Branca|[0.04183304764652...|       1.0|
|Mulher|  Branca|[0.04143394363343...|       1.0|
|Mulher|  Branca|[0.04138723041131...|       1.0|
|Mulher|  Branca|[0.04136389270102...|       1.0|
|Mulher|  Branca|[0.04129395509935...|       1.0|
|Mulher|  Branca|[0.04117764382489...|       1.0|
|Mulher|  Branca|[0.34673282857066...|       1.0|
|Mulher|  Branca|[0.34248039673095...|       1.0|
|Mulher|  Branca|[0.34967068621004...|       1.0|
| Homem|  Branca|[0.53332585897991...|       0.0|
|Mulher|Indigena|[0.09110427023426...|       1.0|
| Homem|  Branca|[0.24369747885579...|       1.0|
|Mulher|  Branca|[0.13057134314797...|       1.0|
|Mulher|  Branca|[0.13070499238089...|       1.0|
| Homem|  Branca|[0.53083549784513...|       0.0|
|Mulher|   Negra|[0.28918635203307...|       1.0|
| Homem|  Branca|[0.52980960134916...|       0.0|


In [22]:
def vec_to_array(v):
    return [float(x) for x in v.toArray()]

vec_to_array_udf = F.udf(vec_to_array, ArrayType(DoubleType()))

In [24]:
pred2 = pred.withColumn("prob_array", vec_to_array_udf(F.col("probability")))
pred2.select("prob_array").show(5, truncate=False)

[Stage 110:==========================================>              (3 + 1) / 4]

+------------------------------------------+
|prob_array                                |
+------------------------------------------+
|[0.041833047646524346, 0.9581669523534757]|
|[0.04143394363343372, 0.9585660563665663] |
|[0.04138723041131609, 0.9586127695886839] |
|[0.04136389270102566, 0.9586361072989743] |
|[0.04129395509935337, 0.9587060449006466] |
+------------------------------------------+
only showing top 5 rows


In [26]:
pred2 = pred2.withColumn("prob_pobreza", F.col("prob_array")[1])
pred2.groupBy("sexo", "raca_cor").agg(F.mean("prob_pobreza").alias("media_prob")) \
     .orderBy("media_prob", ascending=False).show(50)

[Stage 111:=================================================>       (7 + 1) / 8]

+------+--------+------------------+
|  sexo|raca_cor|        media_prob|
+------+--------+------------------+
|Mulher|Indigena|0.9066552447921586|
|Mulher|   Negra|0.8749100237521713|
| Homem|Indigena|0.8315427002121163|
|Mulher|Ignorado|0.8019758276245823|
|Mulher|  Branca|0.7869165075629009|
| Homem|   Negra|0.7825349655339151|
|Mulher| Amarela|0.7800691240242719|
| Homem|  Branca|0.6660755653194937|
| Homem| Amarela|0.6590405250656753|
| Homem|Ignorado|0.6508523905960469|
+------+--------+------------------+



### Persistência da Pobreza ao Longo do Tempo

In [24]:
# criação variável sexo_raça, mesclando ambas as colunas para melhor visualização
pnadc_2023_2025_3 = pnadc_2023_2025_3.withColumn("sexo_raca", F.concat_ws("_", F.col("sexo"), F.col("raca_cor")))

In [27]:
# Calcular taxa de pobreza anual por grupo
pobreza_anual = (pnadc_2023_2025_3.groupBy("ano", "sexo_raca").agg(F.avg("ind_pobreza").alias("taxa_pobreza")).orderBy("ano", "sexo_raca"))
pobreza_anual.show(50, truncate=False)

[Stage 148:=================================================>       (7 + 1) / 8]

+----+---------------+------------------+
|ano |sexo_raca      |taxa_pobreza      |
+----+---------------+------------------+
|2023|Homem_Amarela  |0.721558120362737 |
|2023|Homem_Branca   |0.6996137145435064|
|2023|Homem_Ignorado |0.6739130434782609|
|2023|Homem_Indigena |0.8600397614314115|
|2023|Homem_Negra    |0.8149629421450679|
|2023|Mulher_Amarela |0.8127609821279439|
|2023|Mulher_Branca  |0.8119584188125785|
|2023|Mulher_Ignorado|0.848             |
|2023|Mulher_Indigena|0.9235537190082644|
|2023|Mulher_Negra   |0.9014874040824384|
|2024|Homem_Amarela  |0.6481390793339863|
|2024|Homem_Branca   |0.6598532228267318|
|2024|Homem_Ignorado |0.6190476190476191|
|2024|Homem_Indigena |0.8300055980593394|
|2024|Homem_Negra    |0.7723037981037267|
|2024|Mulher_Amarela |0.7583945956685874|
|2024|Mulher_Branca  |0.7784384385911818|
|2024|Mulher_Ignorado|0.7230769230769231|
|2024|Mulher_Indigena|0.9051012145748988|
|2024|Mulher_Negra   |0.8684350522835232|
|2025|Homem_Amarela  |0.6081424936

In [28]:
# Calcular tendência
w = Window.partitionBy("sexo_raca").orderBy("ano")
pobreza_anual = pobreza_anual.withColumn("taxa_pobreza_anterior", F.lag("taxa_pobreza").over(w))
pobreza_anual = pobreza_anual.withColumn("variacao", F.col("taxa_pobreza") - F.col("taxa_pobreza_anterior"))
pobreza_anual.show(50, truncate=False)

+----+---------------+------------------+---------------------+---------------------+
|ano |sexo_raca      |taxa_pobreza      |taxa_pobreza_anterior|variacao             |
+----+---------------+------------------+---------------------+---------------------+
|2023|Homem_Amarela  |0.721558120362737 |NULL                 |NULL                 |
|2024|Homem_Amarela  |0.6481390793339863|0.721558120362737    |-0.07341904102875074 |
|2025|Homem_Amarela  |0.6081424936386769|0.6481390793339863   |-0.039996585695309395|
|2023|Homem_Branca   |0.6996137145435064|NULL                 |NULL                 |
|2024|Homem_Branca   |0.6598532228267318|0.6996137145435064   |-0.03976049171677465 |
|2025|Homem_Branca   |0.6362024066091954|0.6598532228267318   |-0.023650816217536397|
|2023|Homem_Ignorado |0.6739130434782609|NULL                 |NULL                 |
|2024|Homem_Ignorado |0.6190476190476191|0.6739130434782609   |-0.054865424430641796|
|2025|Homem_Ignorado |0.6515151515151515|0.61904761904

In [30]:
resultado_final = pobreza_anual.join(contagem, on=["ano", "sexo_raca"], how="left")
resultado_final.show(50, truncate=False)

+----+---------------+------------------+---------------------+---------------------+------+
|ano |sexo_raca      |taxa_pobreza      |taxa_pobreza_anterior|variacao             |count |
+----+---------------+------------------+---------------------+---------------------+------+
|2023|Homem_Amarela  |0.721558120362737 |NULL                 |NULL                 |4852  |
|2024|Homem_Amarela  |0.6481390793339863|0.721558120362737    |-0.07341904102875074 |4084  |
|2025|Homem_Amarela  |0.6081424936386769|0.6481390793339863   |-0.039996585695309395|2751  |
|2023|Homem_Branca   |0.6996137145435064|NULL                 |NULL                 |354660|
|2024|Homem_Branca   |0.6598532228267318|0.6996137145435064   |-0.03976049171677465 |354006|
|2025|Homem_Branca   |0.6362024066091954|0.6598532228267318   |-0.023650816217536397|267264|
|2023|Homem_Ignorado |0.6739130434782609|NULL                 |NULL                 |92    |
|2024|Homem_Ignorado |0.6190476190476191|0.6739130434782609   |-0.0548

### Finalização do notebook

In [28]:
executionTime()

Tempo de execucao ate este ponto: 2:50:55.417207
